# Azure Auto ML for Image Data

This notebook is a quick tutorial/guide on how to use Azure AutoML for NLP. You can also checkout this [Microsoft tutorial](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-auto-train-nlp-models?view=azureml-api-2&tabs=python).

*Note: In order to use AutoML for NLP you need to have a **GPU compute cluster**.*

# Notebook Setup

Set project paths and load workspace MLClient.

In [1]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Found the config file in: /config.json


Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


# Prepare Data

Firt we need data. We'll work on [BBC News Dataset](https://www.kaggle.com/datasets/moazeldsokyx/bbc-news) that can be found on kaggle.

In [ ]:
from pathlib import Path
# Set data path
dataset_dir = Path(os.path.join(project_dir, "data/azure-ml-labs-data/nlp-sentiment-analysis"))

## Download and extract the data locally

We will download the Azure example dataset for sentiment analysis.

In [ ]:
import os
import urllib
from zipfile import ZipFile

# Destination folders
training_mltable_path = dataset_dir / "training-mltable-folder"
training_mltable_path.mkdir(exist_ok=True)
validation_mltable_path = dataset_dir / "validation-mltable-folder"
validation_mltable_path.mkdir(exist_ok=True)

# Download dataset files and copy within each MLTable folder

training_download_url = "https://raw.githubusercontent.com/dotnet/spark/main/examples/Microsoft.Spark.CSharp.Examples/MachineLearning/Sentiment/Resources/yelptrain.csv"
training_data_file = training_mltable_path / "yelp_training_set.csv"
urllib.request.urlretrieve(training_download_url, filename=str(training_data_file))

valid_download_url = "https://raw.githubusercontent.com/dotnet/spark/main/examples/Microsoft.Spark.CSharp.Examples/MachineLearning/Sentiment/Resources/yelptest.csv"
valid_data_file = validation_mltable_path / "yelp_validation_set.csv"
urllib.request.urlretrieve(valid_download_url, filename=str(valid_data_file))

print("Dataset files downloaded...")

Dataset URL: https://www.kaggle.com/datasets/moazeldsokyx/bbc-news


## Create Azure Data Asset

You need to prepare the data in 

In [6]:
%%writefile data/azure-ml-labs-data/nlp-sentiment-analysis/training-mltable-folder/MLTable

paths:
  - file: ./yelp_training_set.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'utf8'

Writing data/azure-ml-labs-data/nlp-sentiment-analysis/training-mltable-folder/MLTable


In [7]:
%%writefile data/azure-ml-labs-data/nlp-sentiment-analysis/validation-mltable-folder/MLTable

paths:
  - file: ./yelp_validation_set.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'utf8'

Writing data/azure-ml-labs-data/nlp-sentiment-analysis/validation-mltable-folder/MLTable


In [9]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes


train_data_mltable_asset = Data(
    name="azure-nlp-sentiment-train-mltable",
    path=str(training_mltable_path),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
train_registered_mltable = ml_client.data.create_or_update(train_data_mltable_asset)


valid_data_mltable_asset = Data(
    name="azure-nlp-sentiment-valid-mltable",
    path=str(validation_mltable_path),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
valid_registered_mltable = ml_client.data.create_or_update(valid_data_mltable_asset)

Uploading validation-mltable-folder (0.03 MBs): 100%|██████████| 30847/30847 [00:00<00:00, 1192820.82it/s]




# Running Auto ML for Image Classification

## Load Inputs


Now we can load an MLTable asset we've created earlier.

In [10]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# Training MLTable defined locally, with local data to be uploaded
# my_training_data_input = Input(type=AssetTypes.MLTABLE, path=training_mltable_path)

# Validation MLTable defined locally, with local data to be uploaded
# my_validation_data_input = Input(type=AssetTypes.MLTABLE, path=validation_mltable_path)

my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:azure-nlp-sentiment-train-mltable:1")

# WITH REMOTE PATH: If available already in the cloud/workspace-blob-store
my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:azure-nlp-sentiment-train-mltable:1")
my_validation_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:azure-nlp-sentiment-valid-mltable:1")

## Configure and run the AutoML NLP Text Classification Multiclass training job

In [12]:
# general job parameters
exp_name = "dmdp100-nlp-text-classification-experiment"
dataset_language_code = "eng"

In [13]:
from azure.ai.ml import automl
# Create the AutoML job with the related factory-function.

text_classification_job = automl.text_classification(
    compute="dmdp100-gpu-cluster",
    display_name="dmdp100-nlp-text-classification-multiclass-job-01",
    experiment_name=exp_name,
    training_data=my_training_data_input,
    validation_data=my_validation_data_input,
    target_column_name="Sentiment",
    primary_metric="accuracy",
)
text_classification_job.set_limits(timeout_minutes=120)

In [14]:
text_classification_job.set_featurization(dataset_language=dataset_language_code)

## Run classification job

In [15]:
# Submit the AutoML job

returned_job = ml_client.jobs.create_or_update(
    text_classification_job
)  # submit the job to the backend

print(f"Created job: {returned_job}")

Created job: compute: azureml:dmdp100-gpu-cluster
creation_context:
  created_at: '2025-11-18T15:35:56.697311+00:00'
  created_by: Dominik Mika
  created_by_type: User
display_name: dmdp100-nlp-text-classification-multiclass-job-01
experiment_name: dmdp100-nlp-text-classification-experiment
featurization:
  dataset_language: eng
id: azureml:/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/jobs/mango_rhubarb_jhdhybd7pb
limits:
  max_concurrent_trials: 1
  max_nodes: 1
  max_trials: 1
  timeout_minutes: 120
log_verbosity: info
name: mango_rhubarb_jhdhybd7pb
outputs: {}
primary_metric: accuracy
properties:
  azureml.git.dirty: 'True'
  mlflow.source.git.branch: main
  mlflow.source.git.commit: 240061b50249505bfaf1e3392d627ec643c6cf92
  mlflow.source.git.repoURL: git@github.com:dmika1234/dp100-learn.git
queue_settings:
  job_tier: 'null'
resources:
  instance_count: 1
  shm_size: 2g


In [ ]:
ml_client.jobs.stream(returned_job.name)

# Other

Uncategorized code snippets

In [ ]:
# Create MLTable files
for d in [full_dir, train_dir, valid_dir]:
    (d / "MLTable").write_text(
        """paths:
  - file: ./data.csv"""
    )

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes


full_data_mltable_asset = Data(
    name="bbc-news-mltable",
    path=str(full_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(full_data_mltable_asset)

train_data_mltable_asset = Data(
    name="bbc-news-train-mltable",
    path=str(train_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(train_data_mltable_asset)

valid_data_mltable_asset = Data(
    name="bbc-news-valid-mltable",
    path=str(valid_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(valid_data_mltable_asset)

full_data_mltable_asset = Data(
    name="bbc-news-mltable",
    path=str(full_dir),
    type=AssetTypes.MLTABLE,
    datastore="dmdp100",
    description="BBC News text data prepared for AutoML"
)
registered_mltable = ml_client.data.create_or_update(full_data_mltable_asset)

print("Data assets registered successfully.")


In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

my_data = Data(
    path=str(subset_dir),
    datastore="dmdp100",
    type=AssetTypes.URI_FOLDER,
    description="Subset of Intel Image Classification dataset for AutoML image training",
    name="intel-image-subset-folder",
)
uploaded_data = ml_client.data.create_or_update(my_data)

In [ ]:

# import json

# subset_dir = dataset_dir / "subset"
# jsonl_path = subset_dir / "train_annotations.jsonl"

# records = []

# for class_dir in subset_dir.iterdir():
#     if class_dir.is_dir():
#         label = class_dir.name
#         for img_path in class_dir.glob("*.jpg"):
#             record = {
#                 "image_url": str(img_path.resolve()),  # full path
#                 "label": label
#             }
#             records.append(record)

# # Write to JSONL
# with open(jsonl_path, "w", encoding="utf-8") as f:
#     for r in records:
#         f.write(json.dumps(r) + "\n")

# print(f"✅ JSONL created at: {jsonl_path}")
# print(f"Total records: {len(records)}")

In [ ]:
import json

img_data_asset = ml_client.data.get("intel-image-subset-folder", version="1")
base_uri = img_data_asset.path

annotations_dir = subset_dir.parent / "annotations"
annotations_dir.mkdir(exist_ok=True)
jsonl_path = annotations_dir / "train_annotations.jsonl"
records = []

for class_dir in subset_dir.iterdir():
    if class_dir.is_dir():
        label = class_dir.name
        for img in class_dir.glob("*.jpg"):
            rel = img.relative_to(subset_dir)
            records.append({"image_url": f"{base_uri}{rel.as_posix()}", "label": label})

with open(jsonl_path, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print("✅ JSONL created:", jsonl_path)